# Automating Web Browsers with a Foundry Agent

In this notebook, we will create an AI agent that can **control a web browser** to perform tasks for you. This is different from web search — instead of just looking up information, the agent can actually navigate websites, click buttons, fill in forms, and read page content.

This is incredibly powerful for tasks like:
- Checking live stock prices on financial websites
- Filling out web forms automatically
- Extracting data from pages that require interaction

## Step 1: Install Packages

First, let us install all the Python packages required for this notebook.

In [ ]:
# Install all required packages for Foundry agent development and browser automation
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

## Step 2: Load Configuration

We import all the libraries we need and load configuration values from the `.env` file. For browser automation, we also need the name of the browser automation connection that was set up in your Foundry project.

In [ ]:
# Standard library for environment variables
import os

# Loads settings from a .env file
from dotenv import load_dotenv

# Azure authentication
from azure.identity import DefaultAzureCredential

# Main Foundry project client
from azure.ai.projects import AIProjectClient

# Agent definition type
from azure.ai.projects.models import PromptAgentDefinition

# Browser automation specific classes:
# - BrowserAutomationToolConnectionParameters: points to the browser connection
# - BrowserAutomationToolParameters: wraps the connection parameters
# - BrowserAutomationAgentTool: the actual tool we attach to the agent
from azure.ai.projects.models import BrowserAutomationToolConnectionParameters, BrowserAutomationToolParameters, BrowserAutomationAgentTool

# Load all settings from the .env file
load_dotenv()

# Read the Foundry project URL
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")

# Read the AI model name
my_model = os.getenv("MODEL_DEPLOYMENT_NAME")

# Read the name of the browser automation connection configured in Foundry
browser_connection_name = os.getenv("BROWSER_AUTOMATION_CONNECTION_NAME")

## Step 3: Connect to Foundry

Create the Foundry project client using our endpoint and credentials.

In [ ]:
# Establish a connection to the Foundry project
foundry_client = AIProjectClient(
    endpoint=my_endpoint,
    credential=DefaultAzureCredential(),
)

## Step 4: Set Up the Chat Client

We need a chat client to communicate with the agent once it is created.

In [ ]:
# Get the OpenAI-compatible chat client from our Foundry connection
chat_client = foundry_client.get_openai_client()

## Step 5: Find the Browser Tool Connection

Your Foundry project has a pre-configured browser automation connection. We need to find its unique ID by searching through all available connections and matching it by name. Think of this like looking up a contact in your address book.

In [ ]:
# We will store the connection ID here once we find it
browser_conn_id = ""

# Loop through every connection in our Foundry project
# When we find the one whose name matches our browser connection, save its ID
for connection in foundry_client.connections.list():
    if connection.name == browser_connection_name:
        browser_conn_id = connection.id
        break  # Stop searching once we found the right one

# Confirm we found the connection
print(f"The Browser Automation Tool Connection ID is: {browser_conn_id}")

## Step 6: Configure the Browser Automation Tool

Now we build the browser tool object. This wraps the connection ID in the correct format that Foundry expects. It is a bit like plugging a cable into the right socket — the tool needs to know which browser service to connect to.

In [ ]:
# Build the browser automation tool with the connection we found above
# This creates a tool object that we can attach to an agent
browser_tool = BrowserAutomationAgentTool(
    browser_automation_preview=BrowserAutomationToolParameters(
        connection=BrowserAutomationToolConnectionParameters(
            project_connection_id=browser_conn_id  # Links the tool to the browser service
        )
    )
)

## Step 7: Create the Browser Agent

We now create the agent and give it the browser automation tool. With this tool, the agent can navigate to websites, interact with page elements, and extract information.

In [ ]:
# Create the browser agent with the automation tool attached
browser_agent = foundry_client.agents.create_version(
    agent_name="web-browser-agent",
    definition=PromptAgentDefinition(
        model=my_model,
        tools=[browser_tool],  # Attach the browser tool so the agent can control a browser
        instructions="You are a helpful assistant that automates web browsing tasks to find information for users.",
    )
)

## Step 8: Open a Conversation

Start a new conversation session for the browser agent.

In [ ]:
# Create a new conversation for communicating with the browser agent
chat_session = chat_client.conversations.create()
print(f"Created conversation with id: {chat_session.id}")

## Step 9: Give the Agent a Browser Task

Now we give the agent a detailed, step-by-step task. Because browser automation involves navigating real web pages, it helps to be specific about what the agent should do. Below, we ask it to check a stock price on Yahoo Finance.

In [ ]:
# Define a detailed browser task for the agent
# We give step-by-step instructions so the agent knows exactly what to do on the website
browser_task = """Your goal is to report the percent of Microsoft year-to-date stock price change.
            To do that, go to the website finance.yahoo.com.
            At the top of the page, you will find a search bar.
            Enter the value 'MSFT', to get information about the Microsoft stock price.
            At the top of the resulting page you will see a default chart of Microsoft stock price.
            Click on 'YTD' at the top of that chart, and report the percent value that shows up just below it."""

In [ ]:
# Send the browser task to our agent and wait for the result
# The agent will open a browser, navigate the website, and return its findings
response = chat_client.responses.create(
    conversation=chat_session.id,
    extra_body={
        "agent": {
            "name": "web-browser-agent",
            "type": "agent_reference"
        }
    },
    input=browser_task
)

# Display what the agent found after completing the browser task
print(f"Agent Response: {response.output_text}")